# PCGT on ogbn-arxiv (Large-Scale Experiment)

Run PCGT on ogbn-arxiv (~170K nodes) on Colab/Kaggle GPU.
SGFormer baseline: **72.63 ± 0.13**

**Runtime**: GPU recommended (T4/V100). Full-graph training, ~170K nodes fit in GPU memory.

## 1. Setup

In [1]:
# Mount Google Drive (Colab) or set paths (Kaggle)
import os
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/PCGT'
    DATA_DIR = '/content/PCGT/data/'
elif IN_KAGGLE:
    REPO_ROOT = '/kaggle/working/PCGT'
    DATA_DIR = '/kaggle/working/PCGT/data/'
else:
    REPO_ROOT = '.'
    DATA_DIR = '../data/'

print(f'Platform: {"Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Local"}')
print(f'REPO_ROOT: {REPO_ROOT}')

Mounted at /content/drive
Platform: Colab
REPO_ROOT: /content/PCGT


In [2]:
# Clone repo (skip if already exists)
if not os.path.exists(f'{REPO_ROOT}/large/main.py'):
    !git clone https://github.com/ranjanchoubey/PCGT.git {REPO_ROOT}
else:
    print(f'Repo already at {REPO_ROOT}')

# If on Colab and repo is on Drive, symlink data
if IN_COLAB and os.path.exists('/content/drive/MyDrive/PCGT/data'):
    !ln -sf /content/drive/MyDrive/PCGT/data {REPO_ROOT}/data
    print('Symlinked data from Drive')

Cloning into '/content/PCGT'...
remote: Enumerating objects: 322, done.
remote: Counting objects: 100% (322/322), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 322 (delta 144), reused 300 (delta 125), pack-reused 0 (from 0)
Receiving objects: 100% (322/322), 9.18 MiB | 27.74 MiB/s, done.
Resolving deltas: 100% (144/144), done.


In [4]:
import torch

# Uninstall previous versions to prevent conflicts
!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster --y

# Install compatible versions using the PyG index URL
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install get-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html

# Optionally, install PyTorch Geometric itself
!pip install git+https://github.com/pyg-team/pytorch_geometric.git

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 12.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
ERROR: Could not find a version that satisfies the requirement get-cluster (from versions: none)
ERROR: No matching distribution found for get-cluster
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-zrodce6b
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-zrodce6b
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit f1e6765616f96d9fc5e222c7e18c55d36b497057
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for tor

In [5]:
! pip install torch-sparse

In [6]:
! pip install performer_pytorch

In [7]:
# Verify
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cpu, CUDA: False


In [8]:
os.chdir(f'{REPO_ROOT}/large')
print(f'Working directory: {os.getcwd()}')
!ls *.py

Working directory: /content/PCGT/large
dataset.py     eval.py	load_data.py  main-batch.py  ours.py   partition.py
data_utils.py  gnns.py	logger.py     main.py	     parse.py


## 2. Download ogbn-arxiv
OGB auto-downloads to `data/ogb/`. ~170K nodes, 128 features, 40 classes.

In [9]:
# Pre-download dataset so we can inspect it
from ogb.nodeproppred import NodePropPredDataset
dataset = NodePropPredDataset(name='ogbn-arxiv', root=f'{DATA_DIR}ogb')
data = dataset[0]
print(f'Nodes: {data[0]["num_nodes"]:,}')
print(f'Edges: {data[0]["edge_index"].shape[1]:,}')
print(f'Features: {data[0]["node_feat"].shape[1]}')
print(f'Classes: {dataset.num_classes}')
split = dataset.get_idx_split()
print(f'Train: {len(split["train"]):,}, Val: {len(split["valid"]):,}, Test: {len(split["test"]):,}')

ModuleNotFoundError: No module named 'ogb'

## 3. SGFormer Baseline
Reproduce the paper number: **72.63 ± 0.13** (5 runs)

In [ ]:
# SGFormer baseline — 5 runs (matches run.sh exactly)
!python main.py --method sgformer --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.5 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --seed 123 --runs 5 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR}

## 4. PCGT on ogbn-arxiv

Start with SGFormer's config, add partition attention.
Key parameters to tune: `--num_partitions`, `--graph_weight`, `--partition_method`

In [ ]:
# PCGT Config A1: METIS K=256, graph_weight=0.5 (balanced, same as SGFormer)
!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.5 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions 256 --partition_method metis \
    --seed 123 --runs 3 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR}

In [ ]:
# PCGT Config A2: METIS K=500 (finer partitions, smaller local attention)
!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.5 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions 500 --partition_method metis \
    --seed 123 --runs 3 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR}

In [ ]:
# PCGT Config A3: METIS K=256, graph_weight=0.8 (more GCN, like medium configs)
!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.8 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions 256 --partition_method metis \
    --seed 123 --runs 3 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR}

In [ ]:
# PCGT Config A4: Random partitions K=256 (ablation)
!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.5 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions 256 --partition_method random \
    --seed 123 --runs 3 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR}

## 5. Validation: Best Config × 5 Runs
Once we find the best config above, run 5 runs for the paper.

In [ ]:
# Fill in best config from above and run 5 times
# PCGT 5-run validation (EDIT: fill in best K and graph_weight)
BEST_K = 256          # <-- update after probes
BEST_GW = 0.5         # <-- update after probes
BEST_METHOD = 'metis'  # <-- update after probes

!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight {BEST_GW} \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions {BEST_K} --partition_method {BEST_METHOD} \
    --seed 123 --runs 5 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR}

## 6. Results Summary

In [ ]:
print('='*60)
print('OGBN-ARXIV RESULTS SUMMARY')
print('='*60)
print(f'SGFormer (paper):  72.63 ± 0.13')
print(f'SGFormer (ours):   [fill after cell 7]')
print(f'PCGT A1 (K=256):   [fill after cell 9]')
print(f'PCGT A2 (K=500):   [fill after cell 10]')
print(f'PCGT A3 (gw=0.8):  [fill after cell 11]')
print(f'PCGT A4 (random):  [fill after cell 12]')
print(f'PCGT best (5-run): [fill after cell 14]')
print('='*60)